In [17]:
#1. อ่านข้อมูล
import os
import pandas as pd
 
BASE = r"C:\Users\LENOVO\OneDrive - BUU\Desktop\Data"
INPUT_FILE = os.path.join(BASE, "raw", "Mental.csv")
OUTPUT_FILE = os.path.join(BASE, "processed", "Mental_cleaned.csv")
OUTLIER_FILE = os.path.join(BASE, "processed", "outlier_summary.csv")
 
df = pd.read_csv(INPUT_FILE)



In [18]:
#2. ตรวจภาพรวมและชนิดข้อมูล
print("ขนาดข้อมูล (แถว, คอลัมน์):", df.shape)
df.info()
print(df.head())

ขนาดข้อมูล (แถว, คอลัมน์): (6420, 8)
<class 'pandas.DataFrame'>
RangeIndex: 6420 entries, 0 to 6419
Data columns (total 8 columns):
 #   Column                                                                             Non-Null Count  Dtype  
---  ------                                                                             --------------  -----  
 0   Entity                                                                             6420 non-null   str    
 1   Code                                                                               6150 non-null   str    
 2   Year                                                                               6420 non-null   int64  
 3   Schizophrenia disorders (share of population) - Sex: Both - Age: Age-standardized  6420 non-null   float64
 4   Depressive disorders (share of population) - Sex: Both - Age: Age-standardized     6420 non-null   float64
 5   Anxiety disorders (share of population) - Sex: Both - Age: Age-standardized    

In [19]:
#3. ตรวจและจัดการค่าหาย
print("ค่าหายแต่ละคอลัมน์:")
print(df.isna().sum())

ค่าหายแต่ละคอลัมน์:
Entity                                                                                 0
Code                                                                                 270
Year                                                                                   0
Schizophrenia disorders (share of population) - Sex: Both - Age: Age-standardized      0
Depressive disorders (share of population) - Sex: Both - Age: Age-standardized         0
Anxiety disorders (share of population) - Sex: Both - Age: Age-standardized            0
Bipolar disorders (share of population) - Sex: Both - Age: Age-standardized            0
Eating disorders (share of population) - Sex: Both - Age: Age-standardized             0
dtype: int64


In [20]:
# ดูว่าแถวที่ Code หายคือพื้นที่อะไร (เป็นกลุ่มรวม ไม่ใช่ข้อมูลเสีย)
print("พื้นที่ที่ไม่มี Code:")
print(df.loc[df["Code"].isna(), "Entity"].unique())

พื้นที่ที่ไม่มี Code:
<StringArray>
[            'Africa (IHME GBD)',            'America (IHME GBD)',
               'Asia (IHME GBD)',             'Europe (IHME GBD)',
           'European Union (27)',         'High-income countries',
          'Low-income countries', 'Lower-middle-income countries',
 'Upper-middle-income countries']
Length: 9, dtype: str


In [21]:
# เติม Unknown แทนการลบแถว
df["Code"] = df["Code"].fillna("Unknown")
print("Code ที่ยังหายหลังเติม:", df["Code"].isna().sum())

Code ที่ยังหายหลังเติม: 0


In [22]:
#4. ตรวจข้อมูลซ้ำ
print("แถวซ้ำทั้งแถว:", df.duplicated().sum())
print("ซ้ำที่ Entity + Year:", df.duplicated(subset=["Entity", "Year"]).sum())
df = df.drop_duplicates().reset_index(drop=True)

แถวซ้ำทั้งแถว: 0
ซ้ำที่ Entity + Year: 0


In [23]:
#5. ทำความสะอาดข้อความ และตั้งชื่อคอลัมน์ใหม่ให้สั้น
df["Entity"] = df["Entity"].str.strip()
 
long_cols = df.columns[3:]  # 5 คอลัมน์โรค (ชื่อยาว)
df = df.rename(columns={
    long_cols[0]: "Schizophrenia_pct",
    long_cols[1]: "Depressive_pct",
    long_cols[2]: "Anxiety_pct",
    long_cols[3]: "Bipolar_pct",
    long_cols[4]: "Eating_pct",
})
disease_cols = ["Schizophrenia_pct", "Depressive_pct", "Anxiety_pct",
                "Bipolar_pct", "Eating_pct"]

In [24]:
#6. ตรวจความสมเหตุสมผลของค่า (ต้องเป็นร้อยละ 0-100 และไม่ติดลบ)
print("ค่าติดลบ:", (df[disease_cols] < 0).sum().sum())
print("ค่าเกิน 100:", (df[disease_cols] > 100).sum().sum())
print("ช่วงปี:", df["Year"].min(), "-", df["Year"].max())
print("จำนวนปีต่อพื้นที่:", df.groupby("Entity")["Year"].count().unique())

ค่าติดลบ: 0
ค่าเกิน 100: 0
ช่วงปี: 1990 - 2019
จำนวนปีต่อพื้นที่: [30]


In [25]:
#7. แยกประเทศออกจากกลุ่มรวม (World, Asia, High-income ฯลฯ)
is_group = (df["Code"] == "Unknown") | (df["Entity"] == "World")
df["Entity_type"] = is_group.map({True: "Region/Group", False: "Country"})
print(df.groupby("Entity_type")["Entity"].nunique())


Entity_type
Country         204
Region/Group     10
Name: Entity, dtype: int64


In [26]:
#8. ตรวจค่าผิดปกติด้วยกฎ 1.5 x IQR (เฉพาะประเทศ)
# ตรวจและนับเท่านั้น ไม่ลบ เพราะค่าสูงเป็นค่าจริงของบางประเทศ
countries = df[df["Entity_type"] == "Country"]
rows = []
for c in disease_cols:
    q1, q3 = countries[c].quantile(0.25), countries[c].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((countries[c] < low) | (countries[c] > high)).sum()
    rows.append({"โรค": c, "Q1": q1, "Q3": q3, "IQR": iqr,
                 "เส้นล่าง": low, "เส้นบน": high,
                 "จำนวนค่าผิดปกติ": n_out,
                 "สัดส่วน(%)": round(n_out / len(countries) * 100, 2)})
outlier_table = pd.DataFrame(rows).round(4)
print(outlier_table.to_string(index=False))

              โรค     Q1     Q3    IQR  เส้นล่าง  เส้นบน  จำนวนค่าผิดปกติ  สัดส่วน(%)
Schizophrenia_pct 0.2417 0.2857 0.0440    0.1757  0.3517              180        2.94
   Depressive_pct 3.0615 4.3720 1.3105    1.0958  6.3377               65        1.06
      Anxiety_pct 3.4245 4.5445 1.1200    1.7445  6.2244              334        5.46
      Bipolar_pct 0.5213 0.8575 0.3362    0.0171  1.3617               30        0.49
       Eating_pct 0.0957 0.2437 0.1481   -0.1265  0.4658              331        5.41


In [27]:
#9. จัดเรียง ปัดทศนิยม และบันทึกไฟล์
df[disease_cols] = df[disease_cols].round(6)
df = df.sort_values(["Entity", "Year"]).reset_index(drop=True)
df = df[["Entity", "Code", "Entity_type", "Year"] + disease_cols]
 
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
outlier_table.to_csv(OUTLIER_FILE, index=False, encoding="utf-8-sig")
print("บันทึกแล้ว:", OUTPUT_FILE)

บันทึกแล้ว: C:\Users\LENOVO\OneDrive - BUU\Desktop\Data\processed\Mental_cleaned.csv


In [28]:
#10. ตรวจผลลัพธ์สุดท้าย
check = pd.read_csv(OUTPUT_FILE, keep_default_na=False)
print("ขนาดหลังคลีน:", check.shape)
print("ช่องว่างที่เหลือ:", (check == "").sum().sum())
print(check["Entity_type"].value_counts())

ขนาดหลังคลีน: (6420, 9)
ช่องว่างที่เหลือ: 0
Entity_type
Country         6120
Region/Group     300
Name: count, dtype: int64
